# BM25 Lexical Index, customized for German

Plain whitespace tokenization is weak for German because of:

- **inflection**: `Kündigung`, `gekündigt`, `Kündigungen` are the same concept in different forms and won't match each other as raw strings
- **stopwords**: short function words (`der`, `die`, `und`, `für`, ...) add noise to term-frequency scoring
- **umlauts/eszett**: `ä ö ü ß` need to survive lowercasing untouched

This notebook builds a German-aware tokenizer (stopword removal + Snowball German stemming) and feeds it into BM25, then shows it retrieving a document a naive tokenizer misses because the query uses a different inflected form of the same word.

Run the cells in order.

In [ ]:
%pip install rank_bm25 nltk

In [ ]:
import re

from nltk.stem.snowball import SnowballStemmer
from rank_bm25 import BM25Okapi

# Small, hardcoded stopword list so this notebook doesn't need nltk.download() /
# network access just to run.
GERMAN_STOPWORDS = {
    "der", "die", "das", "den", "dem", "des", "ein", "eine", "einer", "eines",
    "und", "oder", "aber", "doch", "ist", "sind", "war", "waren", "wird", "werden",
    "nicht", "auch", "auf", "im", "in", "an", "am", "zu", "zum", "zur", "von", "vom",
    "mit", "bei", "nach", "aus", "für", "um", "als", "so", "wie", "dass", "sich",
    "es", "er", "sie", "wir", "ihr", "man", "kann", "muss", "hat", "haben",
}

stemmer = SnowballStemmer("german")


def tokenize_de(text: str) -> list[str]:
    tokens = re.findall(r"[a-z\u00e4\u00f6\u00fc\u00df]+", text.lower())
    tokens = [t for t in tokens if t not in GERMAN_STOPWORDS]
    return [stemmer.stem(t) for t in tokens]

In [ ]:
# Sample German business/legal corpus
documents = [
    "Die Kündigung des Vertrags muss schriftlich erfolgen",
    "Der Kunde hat den Vertrag fristgerecht gekündigt",
    "Die Rechnungsnummer finden Sie oben rechts auf der Rechnung",
    "Datenschutzbestimmungen regeln die Verarbeitung personenbezogener Daten",
    "Die Lieferung erfolgt innerhalb von fünf Werktagen nach Zahlungseingang",
    "Kunden können ihre Kontodaten jederzeit im Kundenportal aktualisieren",
]

tokenized_corpus = [tokenize_de(doc) for doc in documents]

print(f"{'document':55s} tokens")
for doc, tokens in zip(documents, tokenized_corpus):
    print(f"{doc[:52]:55s} {tokens}")

In [ ]:
bm25_de = BM25Okapi(tokenized_corpus)


def search(query: str, top_k: int = 3) -> list[tuple[float, str]]:
    scores = bm25_de.get_scores(tokenize_de(query))
    ranked = sorted(zip(scores, documents), key=lambda pair: pair[0], reverse=True)
    return ranked[:top_k]

In [ ]:
query = "Kann ich meinen Vertrag kündigen?"
print(f"Query: {query!r}\ntokens: {tokenize_de(query)}\n")
for score, doc in search(query):
    print(f"{score:.4f}  {doc}")

## Why the German-specific tokenizer matters

The query uses `Vertrag` (nominative) and `kündigen` (infinitive). The naive tokenizer (lowercase + split, no stemming) scores `Der Kunde hat den Vertrag fristgerecht gekündigt` because `vertrag` matches exactly, but scores the more relevant `Die Kündigung des Vertrags muss schriftlich erfolgen` as **0** — `Vertrags` (genitive case) never matches the naive `vertrag` token as a raw string.

The German stemmer reduces `Vertrag`/`Vertrags` to the same stem and `Kündigung`/`kündigen` to the same stem, so the German-aware index ranks the genitive-case document *highest* instead of missing it entirely. Note the Snowball German stemmer still doesn't strip the `ge-...-t` participle prefix, so `gekündigt` stems differently from `kündigen` — a real limitation worth knowing, not every inflected form collapses to one stem.

In [ ]:
def tokenize_naive(text: str) -> list[str]:
    return text.lower().split()


bm25_naive = BM25Okapi([tokenize_naive(doc) for doc in documents])
naive_scores = bm25_naive.get_scores(tokenize_naive(query))
de_scores = bm25_de.get_scores(tokenize_de(query))

print(f"{'document':55s} {'naive':>8s} {'german':>8s}")
for doc, n_score, d_score in sorted(zip(documents, naive_scores, de_scores), key=lambda r: r[2], reverse=True):
    print(f"{doc[:52]:55s} {n_score:8.4f} {d_score:8.4f}")